# 01 · Data overview and cleaning

This notebook checks the structure and quality of the bookings data, documents every cleaning decision and saves a clean file used by the analysis, the model and the Tableau dashboard.

**Dataset:** Hotel Booking Demand (Antonio, Almeida & Nunes, 2019). Bookings of a resort hotel and a city hotel in Portugal, arrivals from July 2015 to August 2017. The data is anonymised: there is no booking ID or guest name.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/hotel_bookings.csv")
print(df.shape)
df.head()

(119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## 1.1 Structure

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

119,390 bookings and 32 columns. Types to fix or handle:
- `agent` and `company` are IDs stored as floats only because they contain NaN: they are categories, not numbers.
- `children` is a float for the same reason.
- `reservation_status_date` is text and must be converted to a date.
- The arrival date is split into year, month name and day: a real date column is built in 1.6.

## 1.2 Missing values

In [3]:
df.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False)

company     112593
agent        16340
country        488
children         4
dtype: int64

| Column | Missing | Meaning | Treatment |
|---|---|---|---|
| `company` | 112,593 (94.3%) | Not a corporate booking | Replaced by a `has_company` flag |
| `agent` | 16,340 (13.7%) | Booked without an agent | Filled with 0 = no agent |
| `country` | 488 (0.4%) | Truly missing | Filled with "Unknown" |
| `children` | 4 | Truly missing | Filled with 0 |

`agent` and `company` are not errors: an empty value carries meaning.

## 1.3 Duplicates

Without a booking ID, two identical rows can be an error **or** two identical rooms booked together (e.g. a group). Before deciding, the impact on the main metric is measured.

In [4]:
n_dup = df.duplicated().sum()
print(f"Duplicate rows: {n_dup} ({n_dup / len(df):.1%})")
print(f"Rows in groups of identical rows (keep=False): {df.duplicated(keep=False).sum()}")

df_dedup = df.drop_duplicates()
print(f"Cancellation rate, all rows:       {df['is_canceled'].mean():.1%}")
print(f"Cancellation rate, no duplicates:  {df_dedup['is_canceled'].mean():.1%}")

Duplicate rows: 31994 (26.8%)
Rows in groups of identical rows (keep=False): 40165
Cancellation rate, all rows:       37.0%
Cancellation rate, no duplicates:  27.5%


In [5]:
dup_rows = df[df.duplicated(keep=False)]
unique_rows = df[~df.duplicated(keep=False)]

print(f"Cancellation rate, duplicated rows: {dup_rows['is_canceled'].mean():.1%}")
print(f"Cancellation rate, unique rows:     {unique_rows['is_canceled'].mean():.1%}")
print()
print(dup_rows["deposit_type"].value_counts(normalize=True).round(3))
print()
print(dup_rows["market_segment"].value_counts(normalize=True).round(3).head())

Cancellation rate, duplicated rows: 58.4%
Cancellation rate, unique rows:     26.2%

deposit_type
No Deposit    0.642
Non Refund    0.356
Refundable    0.002
Name: proportion, dtype: float64

market_segment
Groups           0.416
Offline TA/TO    0.304
Online TA        0.206
Corporate        0.037
Direct           0.035
Name: proportion, dtype: float64


- 31,994 rows (26.8%) are exact copies of another row; in total 40,165 rows belong to 8,171 groups of identical rows.
- Dropping them would cut the cancellation rate from **37.0% to 27.5%**: the duplicates are not random.
- Duplicated rows are cancelled far more often (**58.4% vs 26.2%**) and are mostly **Groups and Offline TA/TO (72%)**, with a large share of "Non Refund" deposits.

This matches how hotels work with tour operators: blocks of identical rooms are booked months ahead and released if they are not sold. Once anonymised, each room of a block becomes an identical row.

**Decision:** all rows are kept (dropping them would remove the largest cancellations) and flagged with `is_duplicate`, so that results can be checked without them.

In [6]:
df["is_duplicate"] = df.duplicated(keep=False).astype(int)
df["is_duplicate"].sum()

np.int64(40165)

## 1.4 Impossible and unusual values

In [7]:
no_guests = (df["adults"] + df["children"].fillna(0) + df["babies"]) == 0
print(no_guests.sum(), "bookings with no guests")
df.loc[no_guests, ["hotel", "is_canceled", "market_segment", "adr", "stays_in_week_nights"]].head()

180 bookings with no guests


,hotel,is_canceled,market_segment,adr,stays_in_week_nights
2224,Resort Hotel,0,Corporate,0.0,3
2409,Resort Hotel,0,Corporate,0.0,0
3181,Resort Hotel,0,Groups,0.0,2
3684,Resort Hotel,0,Groups,0.0,4
3708,Resort Hotel,0,Groups,0.0,4


In [8]:
print(df["adr"].describe())
print()
print((df["adr"] < 0).sum(), "rows with negative adr")
print((df["adr"] == 0).sum(), "rows with adr = 0")
df.sort_values("adr", ascending=False)[["hotel", "adr", "adults", "stays_in_week_nights", "is_canceled"]].head(3)

count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64

1 rows with negative adr
1959 rows with adr = 0


,hotel,adr,adults,stays_in_week_nights,is_canceled
48515,City Hotel,5400.0,2,1,1
111403,City Hotel,510.0,1,1,0
15083,Resort Hotel,508.0,2,1,0


In [9]:
df.loc[df["adr"] == 0, "market_segment"].value_counts()

market_segment
Complementary    680
Online TA        367
Offline TA/TO    332
Groups           252
Direct           240
Corporate         82
Aviation           6
Name: count, dtype: int64

| Case | Rows | Decision |
|---|---|---|
| No adults, children or babies | 180 | **Removed**: a stay with no guests is not possible (mostly Groups/Corporate with `adr` = 0, likely held rooms or entry errors) |
| `adr` = 5,400 (1 night, city hotel) | 1 | **Removed**: 10x the next highest value (510), almost certainly a typo |
| Negative `adr` (−6.38) | 1 | **Removed**: impossible price |
| `adr` = 0 | 1,959 | **Kept** for counts and cancellation rates, **excluded from ADR**: 680 are Complementary (free stays by definition), the rest may be package deals or entry errors |

## 1.5 Apply the cleaning decisions

In [10]:
df_clean = df.copy()

# Impossible rows
no_guests = (df_clean["adults"] + df_clean["children"].fillna(0) + df_clean["babies"]) == 0
bad_adr = (df_clean["adr"] < 0) | (df_clean["adr"] > 5000)
df_clean = df_clean[~no_guests & ~bad_adr]

# Missing values
df_clean["children"] = df_clean["children"].fillna(0).astype(int)
df_clean["country"] = df_clean["country"].fillna("Unknown")
df_clean["agent"] = df_clean["agent"].fillna(0).astype(int)        # 0 = no agent
df_clean["has_company"] = df_clean["company"].notna().astype(int)  # 1 = corporate booking
df_clean = df_clean.drop(columns="company")

# Types
df_clean["reservation_status_date"] = pd.to_datetime(df_clean["reservation_status_date"])

print(len(df), "->", len(df_clean), "rows")
print(df_clean.isna().sum().sum(), "missing values left")

119390 -> 119208 rows
0 missing values left


## 1.6 New columns

In [11]:
df_clean["arrival_date"] = pd.to_datetime(
    df_clean["arrival_date_year"].astype(str) + "-"
    + df_clean["arrival_date_month"] + "-"
    + df_clean["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d",
)
df_clean["total_nights"] = df_clean["stays_in_weekend_nights"] + df_clean["stays_in_week_nights"]
df_clean["revenue"] = df_clean["adr"] * df_clean["total_nights"]

print(df_clean["arrival_date"].min(), "->", df_clean["arrival_date"].max())
print((df_clean["total_nights"] == 0).sum(), "bookings with 0 nights")
df_clean["total_nights"].describe()

2015-07-01 00:00:00 -> 2017-08-31 00:00:00
645 bookings with 0 nights


count    119208.000000
mean          3.426213
std           2.540633
min           0.000000
25%           2.000000
50%           3.000000
75%           4.000000
max          69.000000
Name: total_nights, dtype: float64

- Arrivals from 1 July 2015 to 31 August 2017: only 2016 is a full year, so months are compared, not yearly totals.
- Median stay of 3 nights. 645 bookings (0.5%) have 0 nights (likely day use): kept, with revenue 0. The longest stay (69 nights) is rare but plausible.

## 1.7 Columns that are only known after booking

In [12]:
pd.crosstab(df_clean["reservation_status"], df_clean["is_canceled"])

is_canceled,0,1
reservation_status,,
Canceled,0,42992
Check-Out,75010,0
No-Show,0,1206


`reservation_status` maps one-to-one to `is_canceled` (Canceled and No-Show = 1, Check-Out = 0): it is written at the end of the booking's life. Using it in a model would be **data leakage**. The same applies to `reservation_status_date` and `assigned_room_type` (assigned at check-in). These columns are kept for reporting but excluded from the model. The 1,206 no-shows are counted as cancellations.

## 1.8 Save

In [13]:
df_clean.to_csv("../data/processed/hotel_bookings_clean.csv", index=False)
print(df_clean.shape)

(119208, 36)


## Summary of cleaning decisions

| # | Finding | Decision |
|---|---|---|
| 1 | `children` (4) and `country` (488) missing | Filled with 0 and "Unknown" |
| 2 | Empty `agent`/`company` = no agent/company | `agent` = 0, `company` → `has_company` |
| 3 | 26.8% duplicated rows, mostly group blocks, cancelled 58.4% of the time | Kept and flagged with `is_duplicate` |
| 4 | 180 bookings with no guests | Removed |
| 5 | `adr` negative (1) and 5,400 (1) | Removed |
| 6 | 1,959 bookings with `adr` = 0 | Kept, excluded from ADR |
| 7 | 645 stays with 0 nights | Kept |
| 8 | `reservation_status`, `reservation_status_date`, `assigned_room_type` | Excluded from the model (leakage) |

**Result:** 119,208 clean bookings and 36 columns.